<a href="https://colab.research.google.com/github/MinhPhuc635/focal/blob/main/C%C4%90AI_FOCAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!git clone https://github.com/tomoyoshki/focal.git
%cd /content/focal/src

Mounted at /content/drive


In [3]:
!pip install -q "transformers>=4.56" tsai timm einops scikit-learn pyyaml tqdm huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.1/332.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 896.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.9 MB/s eta 0:00:00


In [10]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [17]:
import os, re, subprocess, time
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

DS  = "nguyenmp/focal-dataset"
OUT = "/content/data"
SKIP_SUFFIX = (".wav", "gps.csv")

# 1) Lấy danh sách toàn bộ file trên Kaggle (có phân trang)
files, token = {}, None
for _ in range(60):
    cmd = ["kaggle", "datasets", "files", DS, "--page-size", "100", "-v"]
    if token:
        cmd += ["--page-token", token]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    before = len(files)
    for line in out.splitlines():
        m = re.match(r"^(MOD_vehicle/[^,\s]+)[,\s]+(\d+)", line)
        if m:
            files[m.group(1)] = int(m.group(2))
    t = re.search(r"Next Page Token = (\S+)", out)
    token = t.group(1) if t else None
    if not token or len(files) == before:
        break

todo = {f: s for f, s in files.items() if not f.endswith(SKIP_SUFFIX)}
print(f"Kaggle có {len(files)} file; sẽ tải {len(todo)} file, khoảng {sum(todo.values())/1e9:.1f} GB")

# 2) Tải từng file, có thử lại và kiểm tra dung lượng
def fetch(item):
    remote, size = item
    local = os.path.join(OUT, remote)
    if os.path.exists(local) and os.path.getsize(local) == size:
        return remote, True
    os.makedirs(os.path.dirname(local), exist_ok=True)
    for _ in range(3):
        subprocess.run(["kaggle", "datasets", "download", "-d", DS, "-f", remote,
                        "-p", os.path.dirname(local), "--unzip"], capture_output=True)
        if os.path.exists(local) and os.path.getsize(local) == size:
            return remote, True
        time.sleep(3)
    return remote, False

with ThreadPoolExecutor(4) as ex:
    results = list(tqdm(ex.map(fetch, todo.items()), total=len(todo)))

failed = [r for r, ok in results if not ok]
print("Xong. Số file lỗi:", len(failed))
for r in failed[:20]:
    print("  lỗi:", r)

Kaggle có 579 file; sẽ tải 496 file, khoảng 17.8 GB


  0%|          | 0/496 [00:00<?, ?it/s]

Xong. Số file lỗi: 327
  lỗi: MOD_vehicle/raw_data/motor2/rs1/enn.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs1/enz.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs1/figure.png
  lỗi: MOD_vehicle/raw_data/motor2/rs1/timestamps.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs2/aud.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs2/ehz.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs2/ene.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs2/enn.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs2/enz.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs2/figure.png
  lỗi: MOD_vehicle/raw_data/motor2/rs2/timestamps.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs3/aud.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs3/ehz.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs3/ene.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs3/enn.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs3/enz.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs3/figure.png
  lỗi: MOD_vehicle/raw_data/motor2/rs3/timestamps.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs7/aud.csv
  lỗi: MOD_vehicle/raw_data/motor2/rs7/ehz.csv


In [18]:
%%writefile models/DINOv3.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel


class DINOv3(nn.Module):
    """Backbone DINOv3 cho FOCAL. Giữ nguyên giao diện của SW_Transformer."""

    def __init__(self, args) -> None:
        super().__init__()
        self.args = args
        self.config = args.dataset_config["DINOv3"]
        self.modalities = args.dataset_config["modality_names"]
        self.locations = args.dataset_config["location_names"]
        self.img_size = self.config["img_size"]
        out_dim = self.config["loc_out_channels"]
        emb_dim = args.dataset_config["FOCAL"]["emb_dim"]

        self.channel_adapters = nn.ModuleDict()  # đổi số kênh -> 3 kênh (RGB giả)
        self.dino_encoders = nn.ModuleDict()     # mỗi modality 1 DINOv3
        self.mod_out_layers = nn.ModuleDict()
        self.mod_projectors = nn.ModuleDict()

        for mod in self.modalities:
            for loc in self.locations:
                in_ch = args.dataset_config["loc_mod_in_freq_channels"][loc][mod]
                self.channel_adapters[f"{loc}__{mod}"] = nn.Conv2d(in_ch, 3, kernel_size=1)

            self.dino_encoders[mod] = AutoModel.from_pretrained(self.config["pretrained_name"]).float()
            hidden = self.dino_encoders[mod].config.hidden_size
            self.mod_out_layers[mod] = nn.Linear(hidden, out_dim)
            self.mod_projectors[mod] = nn.Sequential(
                nn.Linear(out_dim, emb_dim),
                nn.ReLU(),
                nn.Linear(emb_dim, emb_dim),
            )

        self.sample_dim = out_dim * len(self.modalities)
        self.class_layer = nn.Sequential(
            nn.Linear(self.sample_dim, args.dataset_config[args.task]["num_classes"]),
        )

    def extract_mod_features(self, freq_x):
        mod_features = []
        for mod in self.modalities:
            loc_feats = []
            for loc in self.locations:
                x = freq_x[loc][mod]                          # [b, c, i, s]
                x = torch.sign(x) * torch.log1p(x.abs())      # nén giá trị lớn của FFT
                x = self.channel_adapters[f"{loc}__{mod}"](x)  # [b, 3, i, s]
                x = F.interpolate(
                    x, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False
                )                                              # [b, 3, H, W]
                out = self.dino_encoders[mod](pixel_values=x)
                loc_feats.append(out.last_hidden_state[:, 0])  # token CLS
            feat = torch.stack(loc_feats, dim=1).mean(dim=1)
            mod_features.append(self.mod_out_layers[mod](feat))
        return mod_features

    def forward(self, freq_x, class_head=True, proj_head=False):
        mod_features = self.extract_mod_features(freq_x)
        if class_head:
            return self.class_layer(torch.cat(mod_features, dim=1))
        if proj_head:
            return {m: self.mod_projectors[m](f) for m, f in zip(self.modalities, mod_features)}
        return dict(zip(self.modalities, mod_features))

Writing models/DINOv3.py


In [ ]:
#pretrain
%cd /content/focal/src
!python train.py -model=DINOv3 -dataset=MOD -learn_framework=FOCAL -batch_size=64 -tag=dino

In [ ]:
#finetune
!python train.py -model=DINOv3 -dataset=MOD -learn_framework=FOCAL -task=vehicle_classification -stage=finetune -model_weight=/content/drive/MyDrive/CDAI/weights/MOD_DINOv3/exp1_contrastive_FOCAL-dino

In [ ]:
#test
!python test.py -model=DINOv3 -dataset=MOD -stage=finetune -task=vehicle_classification -model_weight=/content/drive/MyDrive/CDAI/weights/MOD_DINOv3/exp1_contrastive_FOCAL-dino